In [1]:
import sys
from pathlib import Path
from uncertainties import ufloat
import country_converter as coco
import matplotlib.pyplot as plt
import pandas as pd
import math

# ---------------------------- Run from Repo Root ----------------------------
BASE_DIR = Path.cwd().parent
sys.path.append(str(BASE_DIR))

# ---------------------------- File Paths ----------------------------
CR_Box_Countries = BASE_DIR / "data" / "CR_Box_Countries_MS.csv"
country_list_csv = BASE_DIR / "data" / "STANDARD_COUNTRY_LIST.csv"
ecw_country = BASE_DIR / "results" / "ECWbyCountry.csv"

from country_pkg import Country

In [2]:
# ------------------------ Functions ----------------------------

def generate_countries_from_multiple_csvs(
    country_csv_path,
    cr_box_csv_path=None,
    ecw_csv_path=None
):
    # ---------------- Main country CSV ----------------
    df = pd.read_csv(country_csv_path, encoding='cp1252')
    required_cols = ['ISO-3', 'Country Name']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV must have a column named '{col}'")
    
    # ---------------- CR Box CSV ----------------
    cr_box_df = None
    if cr_box_csv_path:
        cr_box_df = pd.read_csv(cr_box_csv_path, encoding='cp1252')
        if "Country" not in cr_box_df.columns:
            raise ValueError("CR Box CSV must have a 'Country' column")
        cr_box_df["Country"] = cr_box_df["Country"].apply(Country._cc.convert, to="name_short")
    
    # ---------------- ECW CSV ----------------
    ecw_df = None
    if ecw_csv_path:
        ecw_df = pd.read_csv(ecw_csv_path, encoding='cp1252')
        required_ecw_cols = ['Country Name', 'Country Code']
        for col in required_ecw_cols:
            if col not in ecw_df.columns:
                raise ValueError(f"ECW CSV must have a column named '{col}'")
    
    countries = {}
    
    for _, row in df.iterrows():
        iso_code = row['ISO-3']
        country_name = row['Country Name']
        
        # Create country object
        c = Country(name=iso_code)
        c.properties['ISO-3'] = iso_code
        
        # ---------------- Merge CR Box properties ----------------
        if cr_box_df is not None:
            standardized_name = Country._cc.convert(country_name, to="name_short")
            cr_row = cr_box_df[cr_box_df["Country"] == standardized_name]
            if not cr_row.empty:
                for col in cr_row.columns:
                    if col != "Country":
                        c.properties[col] = cr_row.iloc[0][col]
            else:
                for col in cr_box_df.columns:
                    if col != "Country":
                        c.properties[col] = 0
        
        # ---------------- Merge ECW properties ----------------
        if ecw_df is not None:
            ecw_row = ecw_df[ecw_df["Country Code"] == iso_code]
            if not ecw_row.empty:
                for col in ecw_row.columns:
                    if col != "Country Code":
                        if col != "Country Name":
                            c.properties[col] = ecw_row.iloc[0][col]
        
        countries[iso_code] = c
    
    return countries


In [3]:
## ---------------------------- CR Box Reference ---------------------------- 

Ind_Market_Rev_Per_MERV = {
    '17-20' : 2208.7e6,
    '5-8'   : 563.4e6,
    '9-12'  : 1271.8e6,
    '1-4'   : 171.7e6,
    '13-16' : 1878.1e6}

Tot_Ind_Air_Filter = Ind_Market_Rev_Per_MERV['17-20']+Ind_Market_Rev_Per_MERV['5-8']+Ind_Market_Rev_Per_MERV['1-4']+Ind_Market_Rev_Per_MERV['9-12']+Ind_Market_Rev_Per_MERV['13-16']

Tot_Air_Filter = 20.8303e9

All_Market_Rev_Per_MERV = {
    '17-20' : Ind_Market_Rev_Per_MERV['17-20']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '5-8'   : Ind_Market_Rev_Per_MERV['5-8']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '9-12'  : Ind_Market_Rev_Per_MERV['9-12']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '1-4'   : Ind_Market_Rev_Per_MERV['1-4']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '13-16' : Ind_Market_Rev_Per_MERV['13-16']*Tot_Air_Filter/Tot_Ind_Air_Filter}

Price_Per_Filter = {
    '1-4'   : ufloat(1031.59,   107.26/2),
    '5-8'   : ufloat(1133.85,   447.48/2),
    '9-12'  : ufloat(1302.51,   554.53/2),
    '13-16' : ufloat(1951.25,   593.63/2),
    '17-20' : ufloat(22925.29,  3740.81/2)}

Volume_to_Sale = 0.508*0.508*0.0254

Sales = {
    '1-4'   : All_Market_Rev_Per_MERV['1-4']/(Price_Per_Filter['1-4']*Volume_to_Sale),
    '5-8'   : All_Market_Rev_Per_MERV['5-8']/(Price_Per_Filter['5-8']*Volume_to_Sale),
    '9-12'  : All_Market_Rev_Per_MERV['9-12']/(Price_Per_Filter['9-12']*Volume_to_Sale),
    '13-16' : All_Market_Rev_Per_MERV['13-16']/(Price_Per_Filter['13-16']*Volume_to_Sale),
    '17-20' : All_Market_Rev_Per_MERV['17-20']/(Price_Per_Filter['17-20']*Volume_to_Sale)}

Panel_Filter = ufloat(0.35,     0.35*0.1/2)
Scale_Up_Factor = 1/0.7

Usable_Filters = (Sales['13-16']+Sales['17-20']) * Panel_Filter * Scale_Up_Factor

CR_Box_CADR_LS = 126.13

In [4]:
## ---------------------------- Countries ---------------------------- 

if "countries_dict" not in globals():
    countries_dict = generate_countries_from_multiple_csvs(country_list_csv, CR_Box_Countries, ecw_country)

sum_scale = 0
for country in countries_dict.values():
    msa = country.properties.get("MSA",0)
    mva = country.properties.get("MVA", 0)
    if msa == 1:
        country.properties["Big_6"] = True
        sum_scale += mva
    else:
        country.properties["Big_6"] = False

scale = Usable_Filters/sum_scale
for country in countries_dict.values():
    # --- CR Box Manufacturing ---
    msa = country.properties.get("MSA",0)
    mva = country.properties.get("MVA", 0)
    x = scale * msa * mva
    if x.nominal_value < 50000:
        x = 0
    else:
        x = ufloat(math.floor(x.nominal_value / 4), x.std_dev)
    country.properties["CR Box"] = x
    country.properties["CR Box Weekly"] = country.properties["CR Box"] / 52
    country.properties["CADR CR Box"] = country.properties.get("CR Box", 0) * CR_Box_CADR_LS
    country.properties["CADR CR Box Weekly"] = country.properties["CADR CR Box"] / 52

    # --- CR Box Initial ---
    if country.properties["Big_6"] == True:
        country.properties["CR Box Initial"] = x*0.7
        country.properties["CR Box Initial Weekly"] = x*0.7 / 52
        country.properties["CADR CR Box Initial"] = x*0.7*CR_Box_CADR_LS
        country.properties["CADR CR Box Initial Weekly"] = x*0.7*CR_Box_CADR_LS / 52

    else: 
        country.properties["CR Box Initial"] = 0
        country.properties["CR Box Initial Weekly"] = 0
        country.properties["CADR CR Box Initial"] = 0
        country.properties["CADR CR Box Initial Weekly"] = 0

    # --- Delay Function ---
    m = country.properties.get("MFS", 0)
    if m >= 90:
        country.properties['Distribution Delay'] = 1
    elif 55.5 <= m < 90:
        country.properties['Distribution Delay'] = (443/23) - (14/69) * m
    else:
        country.properties['Distribution Delay'] = 0
    
print(countries_dict['CHN'].summary())

--- China ---
ISO-3: CHN
MFS: 95.5
MVA: 4658790000000.0
MSA: 1.0
Labour Force (2024): 774000000.0
Region: Eastern Asia
%ECW ILO: 0.3667734294194942
%ECW Poll: 0.1526772716333544
ECW ILO: 283882634.37068856
ECW Poll: 118172208.24421635
Big_6: True
CR Box: (2.8+/-1.6)e+07
CR Box Weekly: (5.3+/-3.1)e+05
CADR CR Box: (3.5+/-2.1)e+09
CADR CR Box Weekly: (7+/-4)e+07
CR Box Initial: (1.9+/-1.1)e+07
CR Box Initial Weekly: (3.7+/-2.2)e+05
CADR CR Box Initial: (2.4+/-1.4)e+09
CADR CR Box Initial Weekly: (4.7+/-2.8)e+07
Distribution Delay: 1
None


In [10]:
## ---------------------------- Scale Up ---------------------------- 

output_path = BASE_DIR / "results" / "Scale_up_output_MS.csv"

def scale_up(country,t):
    i = 1
    data_point = [0]
    while i <= t:
        prev = data_point[-1]
        if prev == 0:
            prev = country.properties["CADR CR Box Initial"]
            data_point[-1] = prev
        if country.properties["Distribution Delay"] == None:
            next = 0
        if i<country.properties["Distribution Delay"]:
            next = 0
        else:
            next = prev+ country.properties["CADR CR Box Weekly"]
        data_point.append(next)
        i=i+1
    return data_point

scale_up_data ={}
for country in countries_dict.values():
    t = 52
    data_points = scale_up(country,t)
    scale_up_data[country.name] = data_points

df = pd.DataFrame(scale_up_data).T
df.index.name = "Country"

df.to_csv(output_path, index=True)

               0    1    2    3                4                5   \
Country                                                              
Aruba         0.0  0.0  0.0  0.0              0.0              0.0   
Afghanistan   0.0  0.0  0.0  0.0              0.0              0.0   
Angola        0.0  0.0  0.0  0.0              0.0              0.0   
Albania       0.0  0.0  0.0  0.0              0.0              0.0   
Andorra       0.0  0.0  0.0  0.0              0.0              0.0   
...           ...  ...  ...  ...              ...              ...   
Kosovo        0.0  0.0  0.0  0.0              0.0              0.0   
Yemen         0.0  0.0  0.0  0.0              0.0              0.0   
South Africa    0    0    0    0  (5.6+/-3.3)e+05  (1.1+/-0.7)e+06   
Zambia        0.0  0.0  0.0  0.0              0.0              0.0   
Zimbabwe      0.0  0.0  0.0  0.0              0.0              0.0   

                           6                7                8   \
Country               